# Tema 3B - Vibraciones Mecanicas de Sistemas de 1 GDL

**Asignatura:** Teoria de Maquinas y Mecanismos

---

### Objetivos

1. Plantear la **ecuacion de movimiento** de un sistema de 1 GDL
2. Analizar la **vibracion libre** sin y con amortiguamiento
3. Resolver la **vibracion forzada** armonica y la **respuesta en frecuencia**
4. Calcular la **transmisibilidad** y aplicar el **decremento logaritmico**
5. Aplicar las **ecuaciones de Lagrange** a sistemas vibratorios

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Rectangle, FancyBboxPatch
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp

# ── Paleta de colores ──
COLOR_PRINCIPAL = '#2171b5'
COLOR_FIJO      = '#636363'
COLOR_PUNTO     = '#238b45'
COLOR_ROJO      = '#cb181d'
COLOR_NARANJA   = '#ff7f00'
COLOR_MORADO    = '#6a3d9a'
COLOR_CLARO     = '#a6cee3'

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Formulario del Tema

### Frecuencia natural y amortiguamiento

$$\boxed{\omega_n = \sqrt{\frac{k}{m}}}  \qquad  \boxed{f_n = \frac{\omega_n}{2\pi}}  \qquad  \boxed{T_n = \frac{2\pi}{\omega_n}}$$

$$\boxed{\xi = \frac{c}{2\,m\,\omega_n} = \frac{c}{c_c}}  \qquad  \boxed{c_c = 2\,m\,\omega_n = 2\sqrt{k\,m}}$$

$$\boxed{\omega_d = \omega_n\sqrt{1-\xi^2}} \quad (\text{frecuencia amortiguada, } \xi < 1)$$

### Ecuacion de movimiento (1 GDL)

$$\boxed{m\,\ddot{x} + c\,\dot{x} + k\,x = F(t)}$$

### Vibracion libre sin amortiguamiento ($c = 0$)

$$\boxed{x(t) = X\sin(\omega_n t + \phi)}  \qquad  X = \sqrt{x_0^2 + \left(\frac{\dot{x}_0}{\omega_n}\right)^2}$$

### Vibracion libre amortiguada (subamortiguado $\xi < 1$)

$$\boxed{x(t) = X\,e^{-\xi\omega_n t}\sin(\omega_d t + \phi)}$$

### Decremento logaritmico

$$\boxed{\delta = \ln\frac{x_k}{x_{k+1}} = \frac{2\pi\,\xi}{\sqrt{1-\xi^2}}}  \qquad  \boxed{\xi = \frac{\delta}{\sqrt{4\pi^2 + \delta^2}}}$$

### Vibracion forzada armonica

$$F(t) = F_0\sin(\omega t)$$

$$\boxed{X = \frac{F_0/k}{\sqrt{(1-r^2)^2 + (2\xi r)^2}}}  \qquad  r = \frac{\omega}{\omega_n}$$

$$\boxed{\phi = \arctan\frac{2\xi r}{1-r^2}}$$

### Transmisibilidad

$$\boxed{T_r = \frac{F_T}{F_0} = \frac{\sqrt{1+(2\xi r)^2}}{\sqrt{(1-r^2)^2+(2\xi r)^2}}}$$

### Ecuaciones de Lagrange

$$\boxed{\frac{d}{dt}\left(\frac{\partial T}{\partial \dot{q}_j}\right) - \frac{\partial T}{\partial q_j} + \frac{\partial V}{\partial q_j} + \frac{\partial D}{\partial \dot{q}_j} = Q_j^{nc}}$$

- $T$: energia cinetica, $V$: energia potencial, $D = \frac{1}{2}c\,\dot{q}^2$: funcion de disipacion de Rayleigh

## 2. Ecuacion de Movimiento

Un sistema vibratorio de **1 grado de libertad** se modela con una masa $m$, un resorte lineal $k$ y un amortiguador viscoso $c$.

Aplicando la 2a ley de Newton al diagrama de cuerpo libre:

$$\sum F = m\,\ddot{x} \implies -k\,x - c\,\dot{x} + F(t) = m\,\ddot{x}$$

$$\boxed{m\,\ddot{x} + c\,\dot{x} + k\,x = F(t)}$$

Dividiendo entre $m$:

$$\ddot{x} + 2\xi\omega_n\,\dot{x} + \omega_n^2\,x = \frac{F(t)}{m}$$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
ax.set_xlim(-1, 7)
ax.set_ylim(-1, 6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Sistema Masa-Resorte-Amortiguador (1 GDL)', fontsize=13, fontweight='bold')

# Wall
ax.plot([0, 0], [0, 5], color=COLOR_FIJO, lw=3)
for yy in np.linspace(0, 5, 8):
    ax.plot([-0.4, 0], [yy + 0.3, yy], color=COLOR_FIJO, lw=1.5)

# Spring (zigzag)
spring_x = [0, 0.5, 0.8, 1.2, 1.6, 2.0, 2.4, 2.8, 3.2, 3.5, 4.0]
spring_y = [3.8, 3.8, 4.2, 3.4, 4.2, 3.4, 4.2, 3.4, 4.2, 3.8, 3.8]
ax.plot(spring_x, spring_y, color=COLOR_PRINCIPAL, lw=2)
ax.text(2.0, 4.6, '$k$', fontsize=14, color=COLOR_PRINCIPAL, ha='center', fontweight='bold')

# Damper
ax.plot([0, 1.2], [1.5, 1.5], color=COLOR_MORADO, lw=2)
ax.add_patch(Rectangle((1.2, 1.0), 1.6, 1.0, fill=False, edgecolor=COLOR_MORADO, lw=2))
ax.plot([2.0, 2.0], [1.0, 2.0], color=COLOR_MORADO, lw=2)
ax.plot([2.8, 4.0], [1.5, 1.5], color=COLOR_MORADO, lw=2)
ax.text(2.0, 0.4, '$c$', fontsize=14, color=COLOR_MORADO, ha='center', fontweight='bold')

# Mass
ax.add_patch(Rectangle((4.0, 0.8), 1.8, 3.5, facecolor=COLOR_CLARO, edgecolor=COLOR_PRINCIPAL, lw=2.5))
ax.text(4.9, 2.5, '$m$', fontsize=18, ha='center', va='center', fontweight='bold', color=COLOR_PRINCIPAL)

# Force arrow
ax.annotate('', xy=(7, 2.5), xytext=(5.8, 2.5), arrowprops=dict(arrowstyle='->', color=COLOR_ROJO, lw=2.5))
ax.text(6.6, 3.0, '$F(t)$', fontsize=14, color=COLOR_ROJO, fontweight='bold')

# x arrow
ax.annotate('', xy=(6.2, 0.2), xytext=(4.9, 0.2), arrowprops=dict(arrowstyle='->', color=COLOR_PUNTO, lw=2))
ax.text(5.8, -0.3, '$x(t)$', fontsize=13, color=COLOR_PUNTO, fontweight='bold')

# Ground
ax.plot([-0.6, 7], [-0.5, -0.5], color=COLOR_FIJO, lw=1, ls='--')

plt.tight_layout()
plt.show()

## 3. Vibracion Libre sin Amortiguamiento

Con $c = 0$ y $F(t) = 0$:

$$m\,\ddot{x} + k\,x = 0 \implies \ddot{x} + \omega_n^2\,x = 0$$

Solucion general:

$$x(t) = A\cos(\omega_n t) + B\sin(\omega_n t) = X\sin(\omega_n t + \phi)$$

con condiciones iniciales $x(0) = x_0$, $\dot{x}(0) = v_0$:

$$A = x_0, \quad B = \frac{v_0}{\omega_n}, \quad X = \sqrt{A^2 + B^2}$$

## 4. Vibracion Libre con Amortiguamiento

Con $F(t) = 0$: $\;m\,\ddot{x} + c\,\dot{x} + k\,x = 0$

La ecuacion caracteristica es $m\,s^2 + c\,s + k = 0$ con raices:

$$s_{1,2} = -\xi\omega_n \pm \omega_n\sqrt{\xi^2 - 1}$$

### Tres casos segun $\xi$

| Caso | Condicion | Raices | Comportamiento |
|---|---|---|---|
| **Subamortiguado** | $\xi < 1$ | Complejas conjugadas | Oscilacion decreciente |
| **Criticamente amortiguado** | $\xi = 1$ | Raiz doble real | Retorno mas rapido sin oscilacion |
| **Sobreamortiguado** | $\xi > 1$ | Dos reales distintas | Retorno lento sin oscilacion |

**Subamortiguado** ($\xi < 1$):
$$x(t) = e^{-\xi\omega_n t}\left(A\cos\omega_d t + B\sin\omega_d t\right) = X\,e^{-\xi\omega_n t}\sin(\omega_d t + \phi)$$

**Criticamente amortiguado** ($\xi = 1$):
$$x(t) = (A + B\,t)\,e^{-\omega_n t}$$

**Sobreamortiguado** ($\xi > 1$):
$$x(t) = A\,e^{s_1 t} + B\,e^{s_2 t}$$

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

wn = 2 * np.pi  # omega_n
t = np.linspace(0, 3, 600)
x0, v0 = 1.0, 0.0

# Subamortiguado xi=0.1
xi1 = 0.1
wd1 = wn * np.sqrt(1 - xi1**2)
A1 = x0
B1 = (v0 + xi1 * wn * x0) / wd1
X1 = np.sqrt(A1**2 + B1**2)
x_sub = np.exp(-xi1 * wn * t) * (A1 * np.cos(wd1 * t) + B1 * np.sin(wd1 * t))
env_sub = X1 * np.exp(-xi1 * wn * t)

# Criticamente amortiguado xi=1
xi2 = 1.0
A2 = x0
B2 = v0 + wn * x0
x_crit = (A2 + B2 * t) * np.exp(-wn * t)

# Sobreamortiguado xi=2
xi3 = 2.0
s1 = -xi3 * wn + wn * np.sqrt(xi3**2 - 1)
s2 = -xi3 * wn - wn * np.sqrt(xi3**2 - 1)
A3 = (v0 - s2 * x0) / (s1 - s2)
B3 = x0 - A3
x_over = A3 * np.exp(s1 * t) + B3 * np.exp(s2 * t)

ax.plot(t, x_sub, color=COLOR_PRINCIPAL, lw=2.2, label=f'Subamortiguado ($\\xi={xi1}$)')
ax.plot(t, env_sub, '--', color=COLOR_PRINCIPAL, lw=1, alpha=0.5)
ax.plot(t, -env_sub, '--', color=COLOR_PRINCIPAL, lw=1, alpha=0.5)
ax.plot(t, x_crit, color=COLOR_ROJO, lw=2.2, label=f'Critico ($\\xi={xi2}$)')
ax.plot(t, x_over, color=COLOR_MORADO, lw=2.2, label=f'Sobreamortiguado ($\\xi={xi3}$)')

ax.axhline(0, color='grey', lw=0.5)
ax.set_xlabel('Tiempo (s)', fontsize=12)
ax.set_ylabel('$x(t)$ / $x_0$', fontsize=12)
ax.set_title('Comparacion de los tres casos de amortiguamiento', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xlim(0, 3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Ecuaciones de Lagrange

Para sistemas vibratorios es mas comodo usar **coordenadas generalizadas** y el formalismo de Lagrange.

$$\frac{d}{dt}\left(\frac{\partial T}{\partial \dot{q}_j}\right) - \frac{\partial T}{\partial q_j} + \frac{\partial V}{\partial q_j} + \frac{\partial D}{\partial \dot{q}_j} = Q_j^{nc}$$

- $T = \frac{1}{2}m\,\dot{x}^2$: energia cinetica
- $V = \frac{1}{2}k\,x^2$: energia potencial elastica
- $D = \frac{1}{2}c\,\dot{x}^2$: funcion de disipacion de Rayleigh

**Ejemplo rapido:** Para masa-resorte-amortiguador con $q = x$:

$$\frac{d}{dt}(m\,\dot{x}) - 0 + k\,x + c\,\dot{x} = F(t) \implies m\,\ddot{x} + c\,\dot{x} + k\,x = F(t) \; \checkmark$$

Ventaja: no es necesario calcular las reacciones internas.

## 6. Decremento Logaritmico

Para un sistema subamortiguado, el **decremento logaritmico** mide la tasa de decaimiento entre picos sucesivos:

$$\delta = \ln\frac{x_k}{x_{k+1}} = \frac{2\pi\,\xi}{\sqrt{1-\xi^2}}$$

Si se miden $n$ ciclos entre el pico $k$ y el pico $k+n$:

$$\delta = \frac{1}{n}\ln\frac{x_k}{x_{k+n}}$$

De $\delta$ se obtiene $\xi$:

$$\xi = \frac{\delta}{\sqrt{4\pi^2 + \delta^2}}$$

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

wn = 2 * np.pi * 2  # 2 Hz
xi = 0.08
wd = wn * np.sqrt(1 - xi**2)
x0, v0 = 1.0, 0.0

t = np.linspace(0, 2.5, 1000)
A = x0
B = (v0 + xi * wn * x0) / wd
x_t = np.exp(-xi * wn * t) * (A * np.cos(wd * t) + B * np.sin(wd * t))
X_env = np.sqrt(A**2 + B**2)
envelope = X_env * np.exp(-xi * wn * t)

ax.plot(t, x_t, color=COLOR_PRINCIPAL, lw=1.8, label='$x(t)$')
ax.plot(t, envelope, '--', color=COLOR_ROJO, lw=1.2, label='Envolvente $X e^{-\\xi\\omega_n t}$')
ax.plot(t, -envelope, '--', color=COLOR_ROJO, lw=1.2)

# Mark peaks
Td = 2 * np.pi / wd
peak_times = []
for n_pk in range(10):
    tp = (np.arctan2(A, B) / wd) + n_pk * Td
    if 0 < tp < 2.5:
        peak_times.append(tp)

for i, tp in enumerate(peak_times[:6]):
    xp = np.exp(-xi * wn * tp) * (A * np.cos(wd * tp) + B * np.sin(wd * tp))
    ax.plot(tp, xp, 'o', color=COLOR_PUNTO, ms=7, zorder=5)
    if i < 2:
        ax.annotate(f'$x_{i}$', xy=(tp, xp), xytext=(tp + 0.05, xp + 0.08), fontsize=11, color=COLOR_PUNTO, fontweight='bold')

# Delta annotation
if len(peak_times) >= 2:
    ax.annotate('', xy=(peak_times[1], 0.75), xytext=(peak_times[0], 0.75), arrowprops=dict(arrowstyle='<->', color=COLOR_NARANJA, lw=1.8))
    ax.text((peak_times[0] + peak_times[1]) / 2, 0.80, '$T_d$', fontsize=12, ha='center', color=COLOR_NARANJA, fontweight='bold')

ax.axhline(0, color='grey', lw=0.5)
ax.set_xlabel('Tiempo (s)', fontsize=12)
ax.set_ylabel('$x(t)$', fontsize=12)
ax.set_title(f'Vibracion libre subamortiguada ($\\xi = {xi}$) - Decremento logaritmico', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Vibracion Forzada Armonica

Con excitacion $F(t) = F_0\sin(\omega\, t)$, la respuesta estacionaria es:

$$x_p(t) = X\sin(\omega\,t - \phi)$$

donde la **amplitud** y el **desfase** son:

$$X = \frac{F_0/k}{\sqrt{(1-r^2)^2 + (2\xi r)^2}} \qquad r = \frac{\omega}{\omega_n}$$

$$\phi = \arctan\frac{2\xi r}{1 - r^2}$$

El **factor de amplificacion dinamica** es:

$$H(r) = \frac{X}{F_0/k} = \frac{1}{\sqrt{(1-r^2)^2 + (2\xi r)^2}}$$

**Resonancia** ($r = 1$): $H_{\max} \approx \frac{1}{2\xi}$ (para $\xi$ pequeno)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

r = np.linspace(0.01, 3, 500)
xis = [0.05, 0.1, 0.2, 0.5, 1.0]
colors_xi = [COLOR_ROJO, COLOR_NARANJA, COLOR_PRINCIPAL, COLOR_PUNTO, COLOR_MORADO]

for xi_val, col in zip(xis, colors_xi):
    H = 1 / np.sqrt((1 - r**2)**2 + (2 * xi_val * r)**2)
    phi = np.arctan2(2 * xi_val * r, 1 - r**2)
    ax1.plot(r, H, color=col, lw=2, label=f'$\\xi = {xi_val}$')
    ax2.plot(r, np.degrees(phi), color=col, lw=2)

ax1.set_ylabel('$H(r) = X / (F_0/k)$', fontsize=12)
ax1.set_title('Respuesta en Frecuencia - Amplitud y Fase', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10, loc='upper right')
ax1.set_ylim(0, 12)
ax1.axvline(1, color='grey', ls=':', lw=1, alpha=0.5)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('$r = \\omega / \\omega_n$', fontsize=12)
ax2.set_ylabel('Fase $\\phi$ (grados)', fontsize=12)
ax2.set_ylim(0, 180)
ax2.axvline(1, color='grey', ls=':', lw=1, alpha=0.5)
ax2.axhline(90, color='grey', ls=':', lw=1, alpha=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Transmisibilidad

La **transmisibilidad** mide la fraccion de fuerza que se transmite a la base a traves del muelle y el amortiguador:

$$T_r = \frac{F_T}{F_0} = \frac{\sqrt{1 + (2\xi r)^2}}{\sqrt{(1 - r^2)^2 + (2\xi r)^2}}$$

**Observaciones clave:**
- $r < \sqrt{2}$: el sistema amplifica la fuerza ($T_r > 1$)
- $r = \sqrt{2}$: $T_r = 1$ independientemente de $\xi$
- $r > \sqrt{2}$: **aislamiento** de vibraciones ($T_r < 1$)
- Mas amortiguamiento mejora cerca de resonancia pero **empeora** el aislamiento a altas frecuencias

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

r = np.linspace(0.01, 4, 500)
xis = [0.05, 0.1, 0.2, 0.5, 1.0]
colors_xi = [COLOR_ROJO, COLOR_NARANJA, COLOR_PRINCIPAL, COLOR_PUNTO, COLOR_MORADO]

for xi_val, col in zip(xis, colors_xi):
    Tr = np.sqrt(1 + (2 * xi_val * r)**2) / np.sqrt((1 - r**2)**2 + (2 * xi_val * r)**2)
    ax.plot(r, Tr, color=col, lw=2, label=f'$\\xi = {xi_val}$')

ax.axhline(1, color='grey', ls='--', lw=1)
ax.axvline(np.sqrt(2), color=COLOR_FIJO, ls=':', lw=1.5, alpha=0.7)
ax.text(np.sqrt(2) + 0.05, 8, '$r = \\sqrt{2}$', fontsize=11, color=COLOR_FIJO)
ax.fill_betweenx([0, 12], np.sqrt(2), 4, color=COLOR_PUNTO, alpha=0.06)
ax.text(2.8, 7, 'Zona de\naislamiento', fontsize=11, color=COLOR_PUNTO, ha='center', fontstyle='italic')

ax.set_xlabel('$r = \\omega / \\omega_n$', fontsize=12)
ax.set_ylabel('$T_r = F_T / F_0$', fontsize=12)
ax.set_title('Transmisibilidad', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 10)
ax.set_xlim(0, 4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Ejercicios Resueltos

### Ejercicio 1 - Vibracion libre sin amortiguamiento

Una masa de $m = 5$ kg sujeta a un muelle de $k = 2000$ N/m se desplaza $x_0 = 0.02$ m desde el equilibrio y se suelta sin velocidad inicial.

**Calcular:** frecuencia natural, periodo, y posicion a $t = 0.1$ s.

---

**Datos:** $m = 5$ kg, $k = 2000$ N/m, $x_0 = 0.02$ m, $v_0 = 0$ m/s

**Paso 1:** Frecuencia natural
$$\omega_n = \sqrt{\frac{k}{m}} = \sqrt{\frac{2000}{5}} = 20.000 \text{ rad/s}$$

**Paso 2:** Frecuencia ciclica y periodo
$$f_n = \frac{\omega_n}{2\pi} = \frac{20.000}{2\pi} = 3.183 \text{ Hz}$$
$$T_n = \frac{1}{f_n} = 0.3142 \text{ s}$$

**Paso 3:** Posicion a $t = 0.1$ s
$$x(t) = x_0 \cos(\omega_n t) + \frac{v_0}{\omega_n} \sin(\omega_n t)$$
$$x(0.1) = 0.02 \cdot \cos(20.000 \cdot 0.1) = -0.008323 \text{ m}$$

**Resultado:** $\boxed{x(0.1) = -8.323 \text{ mm}}$

### Ejercicio 2 - Decremento logaritmico

Un sistema vibratorio oscila libremente. Se miden los siguientes desplazamientos en picos sucesivos: $x_0 = 8.5$ mm, $x_5 = 3.2$ mm (5 ciclos despues).

**Calcular:** decremento logaritmico, factor de amortiguamiento $\xi$, y amortiguamiento critico $c_c$ si $m = 2$ kg y $k = 5000$ N/m.

---

**Datos:** $x_0 = 8.5$ mm, $x_5 = 3.2$ mm, $n = 5$ ciclos, $m = 2$ kg, $k = 5000$ N/m

**Paso 1:** Decremento logaritmico
$$\delta = \frac{1}{n} \ln\left(\frac{x_0}{x_n}\right) = \frac{1}{5} \ln\left(\frac{8.5}{3.2}\right) = 0.1954$$

**Paso 2:** Factor de amortiguamiento
$$\xi = \frac{\delta}{\sqrt{4\pi^2 + \delta^2}} = \frac{0.1954}{\sqrt{4\pi^2 + 0.1954^2}} = 0.0311$$

**Paso 3:** Amortiguamiento critico y coeficiente de amortiguamiento
$$\omega_n = \sqrt{\frac{k}{m}} = \sqrt{\frac{5000}{2}} = 50.000 \text{ rad/s}$$
$$c_c = 2 m \omega_n = 2 \cdot 2 \cdot 50.000 = 200.000 \text{ N·s/m}$$
$$c = \xi \cdot c_c = 0.0311 \cdot 200.000 = 6.216 \text{ N·s/m}$$

**Resultado:** $\boxed{\xi = 0.0311, \quad c = 6.216 \text{ N·s/m}}$

### Ejercicio 3 - Vibracion forzada

Un motor de $m = 100$ kg se monta sobre amortiguadores de rigidez total $k = 4 \times 10^5$ N/m y amortiguamiento $c = 2000$ N$\cdot$s/m. El desbalanceo genera una fuerza $F(t) = 500\sin(\omega\,t)$ N a la velocidad de operacion $\omega = 50$ rad/s.

**Calcular:** amplitud de vibracion estacionaria y fuerza transmitida a la base.

---

**Datos:** $m = 100$ kg, $k = 4 \times 10^5$ N/m, $c = 2000$ N·s/m, $F_0 = 500$ N, $\omega = 50$ rad/s

**Paso 1:** Parametros del sistema
$$\omega_n = \sqrt{\frac{k}{m}} = \sqrt{\frac{400000}{100}} = 63.25 \text{ rad/s}$$
$$\xi = \frac{c}{2 m \omega_n} = \frac{2000}{2 \cdot 100 \cdot 63.25} = 0.1581$$
$$r = \frac{\omega}{\omega_n} = \frac{50}{63.25} = 0.7906$$

**Paso 2:** Amplitud estacionaria
$$X_{est} = \frac{F_0}{k} = \frac{500}{400000} = 1.250 \text{ mm}$$
$$X = \frac{X_{est}}{\sqrt{(1-r^2)^2 + (2\xi r)^2}} = 2.7735 \text{ mm}$$
$$\phi = \arctan\left(\frac{2\xi r}{1-r^2}\right) = 33.69°$$

**Paso 3:** Transmisibilidad y fuerza transmitida
$$T_r = \frac{\sqrt{1 + (2\xi r)^2}}{\sqrt{(1-r^2)^2 + (2\xi r)^2}} = 2.2871$$
$$F_T = T_r \cdot F_0 = 2.2871 \cdot 500 = 1143.54 \text{ N}$$

**Resultado:** $\boxed{X = 2.7735 \text{ mm}, \quad F_T = 1143.54 \text{ N}}$

### Ejercicio 4 - Lagrange con pendulo-resorte

Un pendulo de masa $m$ y longitud $l$ tiene un muelle de constante $k$ conectado a distancia $a$ del pivote. Usar Lagrange para obtener la ecuacion de movimiento para pequenas oscilaciones ($\sin\theta \approx \theta$, $\cos\theta \approx 1$).

---

**Coordenada generalizada:** $q = \theta$

**Paso 1:** Energia cinetica
$$T = \frac{1}{2} m l^2 \dot{\theta}^2$$

**Paso 2:** Energia potencial (gravitatoria + elastica)
$$V = -m g l \cos\theta + \frac{1}{2} k (a \sin\theta)^2$$

**Paso 3:** Aproximacion para pequenas oscilaciones ($\sin\theta \approx \theta$, $\cos\theta \approx 1 - \theta^2/2$)
$$T = \frac{1}{2} m l^2 \dot{\theta}^2$$
$$V = -m g l \left(1 - \frac{\theta^2}{2}\right) + \frac{1}{2} k a^2 \theta^2 = -m g l + \frac{1}{2}(m g l + k a^2)\theta^2$$

**Paso 4:** Ecuacion de Lagrange: $\frac{d}{dt}\frac{\partial T}{\partial \dot{\theta}} - \frac{\partial T}{\partial \theta} + \frac{\partial V}{\partial \theta} = 0$
$$m l^2 \ddot{\theta} + (m g l + k a^2)\,\theta = 0$$

**Paso 5:** Frecuencia natural
$$\omega_n = \sqrt{\frac{m g l + k a^2}{m l^2}}$$

---

**Ejemplo numerico:** $m = 2$ kg, $l = 0.5$ m, $k = 800$ N/m, $a = 0.3$ m

$$\omega_n = \sqrt{\frac{2 \cdot 9.81 \cdot 0.5 + 800 \cdot 0.3^2}{2 \cdot 0.5^2}} = 12.791 \text{ rad/s}$$

$$f_n = \frac{\omega_n}{2\pi} = 2.036 \text{ Hz}$$

**Resultado:** $\boxed{\omega_n = 12.791 \text{ rad/s}, \quad f_n = 2.036 \text{ Hz}}$

## 10. Catalogo de Ejercicios por Tipo

| # | Tipo | Descripcion | Datos clave |
|---|---|---|---|
| 1 | **Vibracion libre sin amortiguamiento** | Calcular $\omega_n$, $T_n$, $f_n$ y respuesta temporal | $m$, $k$, $x_0$, $v_0$ |
| 2 | **Decremento logaritmico** | Determinar $\xi$ a partir de picos medidos | $x_k$, $x_{k+n}$, $n$ |
| 3 | **Vibracion forzada armonica** | Amplitud estacionaria, desfase, fuerza transmitida | $m$, $k$, $c$, $F_0$, $\omega$ |
| 4 | **Lagrange aplicado a vibraciones** | Obtener ecuacion de movimiento via energias | $T$, $V$, $D$, coordenada $q$ |
| 5 | **Diseno de aislamiento** | Elegir $k$ y $c$ para $T_r < T_{r,\max}$ a frecuencia dada | $\omega$, $T_{r,\max}$, $m$ |
| 6 | **Sistemas equivalentes** | Reducir sistema a 1 GDL equivalente ($k_{eq}$, $m_{eq}$) | Muelles serie/paralelo, masas con inercia |

### Estrategia general de resolucion

1. **Identificar parametros**: $m$, $k$, $c$ (o equivalentes)
2. **Calcular** $\omega_n$, $\xi$, $\omega_d$
3. **Clasificar** el tipo: libre/forzada, sin/con amortiguamiento
4. **Aplicar condiciones iniciales** para determinar constantes
5. **Evaluar** la respuesta temporal o frecuencial
6. **Verificar unidades** (SI): $[\omega] =$ rad/s, $[c] =$ N$\cdot$s/m, $[k] =$ N/m

In [ ]:
# Simulacion: respuesta libre de un sistema subamortiguado con solve_ivp
def vibration_ode(t, y, wn, xi):
    x, v = y
    return [v, -2 * xi * wn * v - wn**2 * x]

wn_sim = 10  # rad/s
xi_sim = 0.15
x0_sim, v0_sim = 0.05, 0.3  # m, m/s

sol = solve_ivp(vibration_ode, [0, 5], [x0_sim, v0_sim], args=(wn_sim, xi_sim), max_step=0.01, dense_output=True)
t_plot = np.linspace(0, 5, 500)
x_plot = sol.sol(t_plot)[0]
v_plot = sol.sol(t_plot)[1]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(t_plot, x_plot * 1000, color=COLOR_PRINCIPAL, lw=2)
ax1.set_ylabel('$x(t)$ (mm)', fontsize=12)
ax1.set_title(f'Simulacion numerica: $\\omega_n={wn_sim}$ rad/s, $\\xi={xi_sim}$', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(t_plot, v_plot * 1000, color=COLOR_ROJO, lw=2)
ax2.set_xlabel('Tiempo (s)', fontsize=12)
ax2.set_ylabel('$\\dot{x}(t)$ (mm/s)', fontsize=12)
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Tabla Resumen

| Concepto | Formula | Significado |
|---|---|---|
| Frecuencia natural | $\omega_n = \sqrt{k/m}$ | Frecuencia de oscilacion sin amortiguamiento |
| Factor de amortiguamiento | $\xi = c / c_c$ | Relacion con amortiguamiento critico |
| Frecuencia amortiguada | $\omega_d = \omega_n\sqrt{1-\xi^2}$ | Frecuencia real de oscilacion ($\xi < 1$) |
| Amplitud forzada | $X = \frac{F_0/k}{\sqrt{(1-r^2)^2+(2\xi r)^2}}$ | Amplitud estacionaria |
| Transmisibilidad | $T_r = \frac{\sqrt{1+(2\xi r)^2}}{\sqrt{(1-r^2)^2+(2\xi r)^2}}$ | Fraccion de fuerza transmitida |
| Decremento log. | $\delta = 2\pi\xi/\sqrt{1-\xi^2}$ | Medida experimental de $\xi$ |
| Lagrange | $\frac{d}{dt}(\partial T/\partial \dot{q}) - \partial T/\partial q + \partial V/\partial q + \partial D/\partial \dot{q} = Q^{nc}$ | Metodo energetico |
| Aislamiento | $r > \sqrt{2} \implies T_r < 1$ | Condicion para reducir transmision |

---

**Notas finales:**
- Siempre verificar que $\xi < 1$ antes de usar formulas de subamortiguado
- El decremento logaritmico es la forma experimental mas comun de medir $\xi$
- Para aislamiento efectivo, operar con $r > \sqrt{2}$ (baja rigidez, baja frecuencia natural)